# 3D-SynTree: Execution & Training Engine
**Structure-Based Molecular Design via Reaction-Constrained Synthon Assembly**

This notebook acts as an execution wrapper for the `3d-syntree` repository. All modeling,
data processing, training, and checkpointing logic resides strictly within the codebase.

**Pipeline:** Setup → Clone → Dependencies → Assets → GPU Detect → Run Codebase → Status

Before running: add an `HF_TOKEN` secret (write permission) via the Colab/Kaggle secrets
manager (the key icon).

In [ ]:
# CELL 1: Environment & Configuration
import os, json

# 1. Credentials (detect existing or prompt cleanly)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            HF_TOKEN = ""

os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF token {'detected' if HF_TOKEN else 'NOT set (checkpoints stay local)'}.")

# 2. Runtime execution configuration - EDIT THESE to your account.
RUNTIME_CONFIG = {
    "repo_url": "https://github.com/Vtheonly/3d-syntree.git",
    "branch": "main",
    "hf_repo_id": "Vtheonly/3d-syntree-checkpoints",
    "config_override": {
        "huggingface": {
            "enabled": bool(HF_TOKEN),
            "repo_id": "Vtheonly/3d-syntree-checkpoints",
            "push_every_n_epochs": 2
        },
        "training": {
            "time_budget_hours": 11.5,
            "batch_size": 16,
            "accumulate_grad_batches": 2
        }
    }
}

with open("runtime_config.json", "w") as f:
    json.dump(RUNTIME_CONFIG, f, indent=2)
print("Configuration profile initialized.")

In [ ]:
# CELL 2: Clone or Pull Repository
import os

REPO_DIR = "3d-syntree"
REPO_URL = RUNTIME_CONFIG["repo_url"]
BRANCH = RUNTIME_CONFIG["branch"]

if os.path.exists(REPO_DIR):
    print(f"Pulling latest changes in {REPO_DIR}...")
    !cd {REPO_DIR} && git checkout {BRANCH} && git pull
else:
    print(f"Cloning {REPO_URL}...")
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
print("Repository synchronization complete.")

In [ ]:
# CELL 3: Install & Verify Dependencies
print("Installing requirements from repository...")
!pip install --quiet --upgrade pip
!pip install --quiet -r requirements.txt
!pip install --quiet -e .

# Verify critical chemistry & geometry packages
import rdkit
import torch
import torch_geometric
import huggingface_hub

print(f"Dependencies verified: PyTorch {torch.__version__}, "
      f"PyG {torch_geometric.__version__}, RDKit {rdkit.__version__}")

In [ ]:
# CELL 4: Download External Assets (Data & Synthon Catalog)
print("Executing headless data and asset retrieval script...")
!python scripts/download_assets.py \
    --target-dataset crossdocked2020 \
    --synthon-subset 3d-diversity-15k \
    --output-dir ./data

print("External assets and synthon libraries ready.")

In [ ]:
# CELL 5: Hardware & Device Verification
import json
from syntree.utils.hardware import configure_runtime_environment

device_info = configure_runtime_environment()
print("Hardware execution profile:")
print(json.dumps(device_info, indent=2))

assert device_info["device"].startswith("cuda"), \
    "No GPU detected: enable GPU acceleration in Runtime > Change runtime type."

In [ ]:
# CELL 6: Execute Codebase Training / Evaluation Pipeline
print("Launching 3D-SynTree training engine via CLI entrypoint...")

!python main.py \
    --mode train \
    --config configs/train_colab_12h.json \
    --runtime-config ../runtime_config.json \
    --resume-auto

In [ ]:
# CELL 7: Final Status, Benchmark Artifacts & HF Hub Verification
import json, os
from syntree.utils.checkpoint import verify_hf_sync

status = verify_hf_sync(RUNTIME_CONFIG["hf_repo_id"])
print("=== EXECUTION RUN COMPLETE ===")
print(f"Latest Checkpoint: {status['latest_remote_checkpoint']}")
print(f"Total Epochs Completed: {status['epochs_completed']}")
print(f"Hugging Face Sync Verified: {status['sync_ok']}")
if "error" in status:
    print(f"(sync note: {status['error']})")

if os.path.exists("experiments/latest_metrics.json"):
    with open("experiments/latest_metrics.json") as f:
        metrics = json.load(f)
    print("\nValidation Summary:")
    print(json.dumps(metrics, indent=2))